# DEMO - OpenMeteo ETL Pipeline

## 🎯 Purpose
This is the **DEMO/EXECUTION FILE** that runs the ETL pipeline. Use this notebook to execute the pipeline.

## 📁 File Organization
* **This notebook**: DEMO notebook (execution environment)
* **Source code**: `openmeteo_etl_pipeline.py` (ETL logic in Pipelines folder)

The notebook imports the `.py` file from the Pipelines subdirectory.

## 📦 For Blackboard Submission

**Option 1: Maintain Folder Structure** (Recommended)
- Upload to Databricks maintaining the structure:
  - `DEMO - OpenMeteo ETL Pipeline.ipynb` (in your home directory)
  - `Pipelines/openmeteo_etl_pipeline.py` (in Pipelines subfolder)
- The notebook already points to the correct relative path

**Option 2: Same Directory**
- Place both files in the same folder
- Update Cell 6, line 4 to use relative import:
  ```python
  sys.path.insert(0, '.')
  ```

## 🚀 What This Does
1. Installs all required Python packages automatically
2. Imports the `.py` module from the Pipelines directory
3. Calls the Open-Meteo API with **fresh data** (today + 7 days)
4. Transforms weather, soil, air quality, and pollen data with derived metrics
5. Loads results to `workspace.weather.hourly_merged` in Unity Catalog

## 📊 Pipeline Output
* **144 rows** (6 days × 24 hours of forecast data)
* **37 columns** including:
  - Weather metrics (temperature, humidity, precipitation, wind)
  - Soil data (temperature, moisture)
  - Air quality (PM2.5, PM10, AQI, gases)
  - Pollen counts (6 allergen types)
  - Derived scores (planting_readiness, allergy_risk)
  - Risk flags (heat_stress, respiratory_risk, best_overall_day, etc.)

## ⚡ How to Run
1. Upload both files to Databricks (maintain folder structure)
2. Click **Run All** or execute cells sequentially
3. The pipeline pulls fresh API data and saves to Unity Catalog

## ⚠️ Important
* Do NOT run the `.py` file directly - it will fail with `ModuleNotFoundError`
* Always use this notebook to execute the pipeline
* The pipeline pulls fresh data from the API on every run (no cached/fixed datasets)
* Location: Louisville, KY (38.2527°N, -85.7585°W)

In [0]:
%pip install openmeteo-requests requests-cache retry-requests numpy pandas sqlalchemy psycopg2-binary

# Restart Python kernel to load newly installed packages
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Restart Python to clear module cache and load updated packages
dbutils.library.restartPython()

In [0]:
# Verify that the .py file is in the Pipelines directory
import os

# Get current notebook path and construct Pipelines path
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
user_home = os.path.dirname(notebook_path)

# Construct the full workspace path to the .py file
if user_home.startswith('/Users/'):
    pipelines_dir = f'/Workspace{user_home}/Pipelines'
else:
    pipelines_dir = f'{user_home}/Pipelines'

py_file_path = os.path.join(pipelines_dir, 'openmeteo_etl_pipeline.py')

print("🔍 Setup Verification")
print("=" * 60)
print(f"📂 Notebook location: {notebook_path}")
print(f"📝 Looking for: {py_file_path}")
print()

# Check if .py file exists
try:
    # Try to read the file
    with open(py_file_path, 'r') as f:
        content = f.read()
        if 'def run_pipeline' in content:
            print("✅ SUCCESS: openmeteo_etl_pipeline.py found!")
            print(f"   • File location: {py_file_path}")
            print(f"   • File size: {len(content):,} characters")
            print(f"   • Contains run_pipeline function: ✓")
            print()
            print("✅ You're ready to run the pipeline!")
            print("   → Proceed to the next cells to execute.")
        else:
            print("⚠️ WARNING: File found but doesn't contain expected functions.")
            print("   Make sure you uploaded the correct openmeteo_etl_pipeline.py file.")
except FileNotFoundError:
    print("❌ ERROR: openmeteo_etl_pipeline.py NOT FOUND")
    print()
    print("🛠️ To fix this:")
    print("   1. Make sure openmeteo_etl_pipeline.py is uploaded to:")
    print(f"      {pipelines_dir}/")
    print("   2. Create a 'Pipelines' subfolder in your user home directory")
    print("   3. Upload the .py file to that folder")
    print()
    print("⚠️ Cannot proceed until the .py file is in the correct location.")

🔍 Setup Verification
📂 Notebook location: /Users/mirandapachini@gmail.com/DEMO - OpenMeteo ETL Pipeline
📝 Looking for: /Workspace/Users/mirandapachini@gmail.com/Pipelines/openmeteo_etl_pipeline.py

✅ SUCCESS: openmeteo_etl_pipeline.py found!
   • File location: /Workspace/Users/mirandapachini@gmail.com/Pipelines/openmeteo_etl_pipeline.py
   • File size: 30,472 characters
   • Contains run_pipeline function: ✓

✅ You're ready to run the pipeline!
   → Proceed to the next cells to execute.


In [0]:
%sql
-- Create the schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS workspace.weather
COMMENT 'Weather and air quality forecast data from Open-Meteo API'

In [0]:
import sys
import os

# Get current notebook path and construct Pipelines path
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
user_home = os.path.dirname(notebook_path)

# Construct the full workspace path to the Pipelines directory
if user_home.startswith('/Users/'):
    pipelines_path = f'/Workspace{user_home}/Pipelines'
else:
    pipelines_path = f'{user_home}/Pipelines'

sys.path.insert(0, pipelines_path)

from openmeteo_etl_pipeline import run_pipeline

# Run the ETL pipeline and load to Unity Catalog
result_df = run_pipeline(load_destination="unity_catalog", table_name="workspace.weather.hourly_merged")

# Display results
if result_df is not None:
    print(f"✅ Pipeline completed successfully!")
    print(f"📊 Final dataset: {result_df.shape[0]} rows × {result_df.shape[1]} columns")
    print(f"📍 Saved to: workspace.weather.hourly_merged")
    print(f"\n🔍 Sample data preview:")
    display(result_df.head(10))
    
    print(f"\n📈 Key Metrics Summary:")
    print(f"   • Average Temperature: {result_df['temperature_2m'].mean():.1f}°F")
    print(f"   • Average Air Quality Index: {result_df['us_aqi'].mean():.1f}")
    print(f"   • Days with Best Overall Conditions: {result_df['best_overall_day_flag'].sum()}")
    print(f"   • Hours with High Pollen: {result_df['high_pollen_flag'].sum()}")
else:
    print("❌ Pipeline failed. Check logs above for details.")

/Workspace/Users/mirandapachini@gmail.com/Pipelines/openmeteo_etl_pipeline.py:442: FutureWarning: Treating datetime data as categorical rather than numeric in `.describe` is deprecated and will be removed in a future version of pandas. Specify `datetime_is_numeric=True` to silence this warning and adopt the future behavior now.
  logging.info("Merged DataFrame Summary Statistics:\n%s", merged.describe(include="all"))


✅ Pipeline completed successfully!
📊 Final dataset: 144 rows × 37 columns
📍 Saved to: workspace.weather.hourly_merged

🔍 Sample data preview:


date,temperature_2m,relative_humidity_2m,precipitation_probability,precipitation,wind_speed_10m,soil_temperature_0cm,soil_moisture_0_to_1cm,pm10,pm2_5,carbon_monoxide,ozone,nitrogen_dioxide,sulphur_dioxide,us_aqi,us_aqi_pm2_5,us_aqi_pm10,us_aqi_nitrogen_dioxide,us_aqi_carbon_monoxide,us_aqi_ozone,us_aqi_sulphur_dioxide,grass_pollen,ragweed_pollen,olive_pollen,mugwort_pollen,birch_pollen,alder_pollen,planting_readiness,allergy_risk,high_wind_flag,rain_expected_flag,soil_too_wet_flag,poor_air_quality_flag,high_pollen_flag,heat_stress_flag,respiratory_risk_flag,best_overall_day_flag
2026-05-31T00:00:00.000Z,65.3666,59.0,0.0,0.0,5.378113,64.8572,0.269,3.8,3.7,131.0,66.0,3.7,1.2,43.77319,19.218748,4.325758,1.822301,1.5072463,43.77319,0.65430754,null,null,null,null,null,null,63.671486,16.785715,0,0,0,0,0,0,0,0
2026-05-31T01:00:00.000Z,64.1066,58.0,0.0,0.0,5.524982,63.5072,0.27,3.8,3.7,130.0,63.0,3.6,1.2,40.70037,18.072916,4.064394,1.7730497,1.4685992,40.70037,0.65430754,null,null,null,null,null,null,61.684616,16.142857,0,0,0,0,0,0,0,0
2026-05-31T02:00:00.000Z,63.296597,59.0,0.0,0.0,3.5370073,62.427197,0.27,3.7,3.7,130.0,61.0,3.6,1.2,37.627552,17.100693,3.84091,1.7730497,1.4299517,37.627552,0.65430754,null,null,null,null,null,null,62.232586,15.714285,0,0,0,0,0,0,0,0
2026-05-31T03:00:00.000Z,61.676598,65.0,0.0,0.0,4.6548953,61.077198,0.271,3.8,3.7,130.0,60.0,3.7,1.3,34.902596,16.249998,3.6477275,1.822301,1.3900965,34.902596,0.70883316,null,null,null,null,null,null,59.2747,15.5,0,0,0,0,0,0,0,0
2026-05-31T04:00:00.000Z,60.8666,66.0,0.0,0.0,4.244409,59.9072,0.272,3.8,3.7,131.0,59.0,3.8,1.3,32.69944,15.52083,3.484849,1.8715523,1.347826,32.69944,0.70883316,null,null,null,null,null,null,58.08519,15.285713,0,0,0,0,0,0,0,0
2026-05-31T05:00:00.000Z,59.966602,68.0,0.0,0.0,5.3688,58.9172,0.273,4.0,3.9,132.0,57.0,3.9,1.4,31.018087,14.947913,3.3560612,1.9208038,1.3091788,31.018087,0.76335883,null,null,null,null,null,null,55.6008,15.0,0,0,0,0,0,0,0,0
2026-05-31T06:00:00.000Z,58.7966,71.0,0.0,0.0,5.3172884,58.3772,0.273,4.1,4.0,134.0,52.0,3.9,1.4,29.626625,14.583333,3.2727275,1.9208038,1.281401,29.626625,0.76335883,null,null,null,null,null,null,54.932312,14.0,0,0,0,0,0,0,0,0
2026-05-31T07:00:00.000Z,58.9766,75.0,0.0,0.0,3.607054,58.4672,0.274,4.0,3.9,136.0,46.0,3.8,1.5,28.293137,14.392361,3.227273,1.8715523,1.2705315,28.293137,0.8178845,null,null,null,null,null,null,56.72255,12.642858,0,0,0,0,0,0,0,0
2026-05-31T08:00:00.000Z,61.8566,70.0,0.0,0.0,4.58994,62.877197,0.274,3.9,3.9,138.0,44.0,3.6,1.5,26.90167,14.288195,3.200758,1.7730497,1.2729468,26.90167,0.8178845,null,null,null,null,null,null,61.619656,12.214286,0,0,0,0,0,0,0,0
2026-05-31T09:00:00.000Z,65.1866,61.0,0.0,0.0,3.5370076,68.5472,0.273,4.0,3.9,139.0,48.0,3.0,1.5,25.62616,14.375003,3.2121215,1.4775414,1.281401,25.62616,0.8178845,null,null,null,null,null,null,70.2726,13.071428,0,0,0,0,0,0,0,1



📈 Key Metrics Summary:
   • Average Temperature: 69.5°F
   • Average Air Quality Index: 34.9
   • Days with Best Overall Conditions: 56
   • Hours with High Pollen: 0
